# 11_Gold_Observation

Gold-layer notebook. Attach `FHIR_Lakehouse` before running.

In [1]:
from pyspark.sql.functions import col, element_at, regexp_extract, to_timestamp, current_timestamp
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, 29a225cd-d085-4d28-9d41-b9667ba46ada, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
silver_observation_df = spark.table("silver.observation")
print("Silver Observation Records:", silver_observation_df.count())

StatementMeta(, 29a225cd-d085-4d28-9d41-b9667ba46ada, 4, Finished, Available, Finished, False)

Silver Observation Records: 27521


In [3]:
gold_observation_df = silver_observation_df.select(
    col("id").alias("observation_id"),
    col("subject.reference").alias("patient_reference"),
    regexp_extract(col("subject.reference"), r"Patient/([^/]+)", 1).alias("patient_id"),
    col("encounter.reference").alias("encounter_reference"),
    col("status").alias("observation_status"),
    col("code.text").alias("observation_name"),
    element_at(col("code.coding"), 1).getField("code").alias("observation_code"),
    element_at(col("code.coding"), 1).getField("display").alias("observation_display"),
    to_timestamp(col("effectiveDateTime")).alias("effective_datetime"),
    to_timestamp(col("issued")).alias("issued_datetime"),
    col("valueQuantity.value").alias("value"),
    col("valueQuantity.unit").alias("unit"),
    col("valueQuantity.code").alias("unit_code"),
    col("valueString").alias("value_string"),
    col("valueInteger").alias("value_integer"),
    col("meta.lastUpdated").alias("source_last_updated"),
    col("extraction_timestamp"),
    col("load_timestamp")
).withColumn("gold_load_timestamp", current_timestamp())

StatementMeta(, 29a225cd-d085-4d28-9d41-b9667ba46ada, 5, Finished, Available, Finished, False)

In [4]:
(gold_observation_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable("gold.fact_observation"))

StatementMeta(, 29a225cd-d085-4d28-9d41-b9667ba46ada, 6, Finished, Available, Finished, False)

In [5]:
spark.table("gold.fact_observation").printSchema()
spark.table("gold.fact_observation").show(10, truncate=False)
print("Total Gold Observation Records:", spark.table("gold.fact_observation").count())

StatementMeta(, 29a225cd-d085-4d28-9d41-b9667ba46ada, 7, Finished, Available, Finished, False)

root
 |-- observation_id: string (nullable = true)
 |-- patient_reference: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_reference: string (nullable = true)
 |-- observation_status: string (nullable = true)
 |-- observation_name: string (nullable = true)
 |-- observation_code: string (nullable = true)
 |-- observation_display: string (nullable = true)
 |-- effective_datetime: timestamp (nullable = true)
 |-- issued_datetime: timestamp (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- unit_code: string (nullable = true)
 |-- value_string: string (nullable = true)
 |-- value_integer: long (nullable = true)
 |-- source_last_updated: string (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- gold_load_timestamp: timestamp (nullable = true)

+--------------+-----------------+----------+-------------------+------------------+------------